# Hierarchical ReLU-LoRA vs Baselines — Qwen1.5-MoE (EMNLP Version)

**Direct port of `hrlora_olmoe_fixed-6.ipynb` to Qwen1.5-MoE-A2.7B.**

## Key differences from OLMoE notebook

- Model: `Qwen/Qwen1.5-MoE-A2.7B` (60 routing experts, top-4, shared always-on expert)
- `down_proj` shape: `[60, out_f=2048, in_f=1408]` → `lora_dim = out_f = 2048` (same value as OLMoE)
- Target layer: **0** (highest top-K Jaccard = 0.154, from pre-run Qwen Jaccard diagnostic)
- Target expert: **5** (highest conflict score = 0.0236)
- Shared expert (`mlp.shared_expert`) is **NOT** wrapped — fires independently on every token
- No AlignDevicesHook needed (single A100)
- `trust_remote_code=True` + `gradient_checkpointing_disable()` required

## What is identical to OLMoE

- All class definitions: `HierarchicalExpert`, `DRLoRALayer`, `ConflictSaturationMonitor`,
  `DRLoRATracker`, `DRLoRAGrowthSchedule`, `perform_rank_growth`
- Dataset pipeline: bigcode/the-stack-smol (Python) vs qiaojin/PubMedQA
- Conflict-scaling grid: Runs A (0%), B (20%), C (50%)
- Fixed-step spawning: `[150, 400, 650, 900, 1100]`
- All hyperparameters: `base_rank=8`, `lr=5e-5`, `n_steps=1500`
- Results table, plots, HumanEval evaluation


## Section 1 — Install & Model Load


In [ ]:
!pip install transformers>=4.41.0 accelerate datasets peft torch>=2.2.0 tabulate human_eval -q

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch
import torch.nn as nn
import torch.nn.functional as F
import gc
import os
import math
import json
from collections import deque, defaultdict
from typing import Dict, List, Optional, Tuple

torch.cuda.empty_cache()
gc.collect()

# GPU check
if not torch.cuda.is_available():
    raise RuntimeError(
        "No GPU detected. "
        "Go to: Runtime -> Change runtime type -> Hardware accelerator -> A100 (or T4). "
        "Then re-run all cells."
    )
print(f"GPU: {torch.cuda.get_device_name(0)}  |  "
      f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

MODEL_ID = "Qwen/Qwen1.5-MoE-A2.7B"

if os.path.exists("/content"):
    SAVE_ROOT = "/content/qwen_hrlora_checkpoints"
    print("Detected: Colab")
else:
    SAVE_ROOT = "./qwen_hrlora_checkpoints"
    print("Detected: Local")

OFFLOAD_DIR = os.path.join(SAVE_ROOT, "offload")
os.makedirs(SAVE_ROOT, exist_ok=True)
os.makedirs(OFFLOAD_DIR, exist_ok=True)
print(f"Checkpoints: {SAVE_ROOT}")

print("")
print(f"Loading {MODEL_ID}...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model_qwen = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.bfloat16,
    device_map="cuda:0",
    offload_folder=OFFLOAD_DIR,
    trust_remote_code=True,
)
model_qwen.gradient_checkpointing_disable()  # required with custom wrappers
model_qwen.config.use_cache = False          # required for gradient flow during training

print("")
print("Model loaded!")
print(f"  Layers:          {model_qwen.config.num_hidden_layers}")
print(f"  Routing experts: {model_qwen.config.num_experts}")
print(f"  Top-k:           {model_qwen.config.num_experts_per_tok}")
print(f"  model_dim:       {model_qwen.config.hidden_size}")
vram = torch.cuda.memory_allocated() / 1e9
print(f"  VRAM:            {vram:.2f} GB")


In [ ]:
# Set your HuggingFace token here — needed for bigcode/the-stack-smol
# Get yours at: https://huggingface.co/settings/tokens

import os
from huggingface_hub import login

HF_TOKEN = "hf_xxxxxxxxxxxxxxxxxxxx"  # <-- paste your token here

os.environ["HF_TOKEN"] = HF_TOKEN
login(token=HF_TOKEN, add_to_git_credential=False)
print("HuggingFace login successful.")

## Section 2 — Class Definitions


In [ ]:
# HierarchicalExpert — Base LoRA sub-adapter module
# Identical to OLMoE implementation — architecture-agnostic.
# Operates on model_dim (2048) for both OLMoE and Qwen.

class HierarchicalExpert(nn.Module):
    """
    Low-rank LoRA sub-adapter. Computes: L(x) = (x A^T B^T) * scaling
    B is zero-initialized: at spawn, contribution is exactly zero (zero-loss-spike guarantee).
    """
    def __init__(self, in_features, out_features, base_rank=16, lora_alpha=32):
        super().__init__()
        self.in_features = in_features
        self.out_features = out_features
        self.rank = base_rank
        self.scaling = lora_alpha / base_rank

        self.A = nn.Parameter(torch.empty(base_rank, in_features))
        nn.init.kaiming_uniform_(self.A, a=math.sqrt(5))
        self.B = nn.Parameter(torch.zeros(out_features, base_rank))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        A = self.A.to(x.dtype)
        B = self.B.to(x.dtype)
        return (x @ A.t() @ B.t()) * self.scaling

print("HierarchicalExpert defined. (identical to OLMoE)")

In [ ]:
# DRLoRALayer — DR-LoRA with capacity reservation and binary rank mask
# Identical to OLMoE implementation.

class DRLoRALayer(nn.Module):
    """
    DR-LoRA Layer with Capacity Reservation.
    Allocates full rank space (r_max) upfront, activates only r_init initially.
    """
    def __init__(self, in_features, out_features, r_max=16, r_init=4,
                 lora_alpha=16, lora_dropout=0.0):
        super().__init__()
        self.in_features = in_features
        self.out_features = out_features
        self.r_max = r_max
        self.r_init = r_init
        self.lora_alpha = lora_alpha

        self.lora_A = nn.Parameter(torch.zeros(r_max, in_features))
        self.lora_B = nn.Parameter(torch.zeros(out_features, r_max))
        self.lora_dropout = nn.Dropout(p=lora_dropout)

        self.register_buffer('rank_mask', torch.zeros(r_max, dtype=torch.bool))
        self.rank_mask[:r_init] = True
        self.active_ranks = r_init
        self.reset_parameters()

    def reset_parameters(self):
        nn.init.kaiming_uniform_(self.lora_A, a=math.sqrt(5))
        nn.init.zeros_(self.lora_B)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        active_mask = self.rank_mask.to(self.lora_A.device)
        if active_mask.any():
            A_active = self.lora_A[active_mask, :].to(x.dtype)
            B_active = self.lora_B[:, active_mask].to(x.dtype)
            active_ranks = active_mask.sum().item()
            scaling = self.lora_alpha / active_ranks
            result = self.lora_dropout(x) @ A_active.T @ B_active.T
            return result * scaling
        return torch.zeros(*x.shape[:-1], self.out_features, device=x.device, dtype=x.dtype)

    def activate_rank(self, rank_idx: int):
        if rank_idx < self.r_max:
            self.rank_mask[rank_idx] = True
            self.active_ranks = self.rank_mask.sum().item()

    def get_active_ranks(self) -> int:
        return int(self.rank_mask.sum().item())

    def get_mask_status(self) -> str:
        mask_str = ''.join(['█' if m else '░' for m in self.rank_mask])
        return f"[{mask_str}] ({self.get_active_ranks()}/{self.r_max})"

print("DRLoRALayer defined. (identical to OLMoE)")

In [ ]:
# ConflictSaturationMonitor — OLMoE version (gradient cosine similarity)
# Identical to OLMoE implementation. Uses gradient cosine similarity instead of
# loss EMA divergence — more robust at 50% conflict where domain losses converge.

class ConflictSaturationMonitor:
    """
    Fires when BOTH hold for `window` consecutive steps:
      1. Plateau: rank importance slope < tau_plateau
      2. Conflict: gradient cosine similarity < cosine_threshold

    Uses gradient cosine similarity (not loss EMA) because at 50% conflict
    both domain losses converge, making EMA divergence useless as a signal.
    """
    def __init__(self, tau_plateau=1e-4, delta_threshold=1.0, window=15, beta=0.9):
        self.tau_plateau      = tau_plateau
        self.delta_threshold  = delta_threshold  # kept for API compat
        self.window           = window
        self.beta             = beta

        self._ri_history: list = []
        self._ema_code    = None
        self._ema_medical = None
        self._plateau_window  = []
        self._conflict_window = []
        self._step_count  = 0

        self._mean_grad_code = None
        self._mean_grad_med  = None
        self.cosine_threshold = 0.1

    def update(self, lora_A, lora_B, lora_B_grad, loss_val, domain):
        """Call once per step with the SINGLE target expert's A, B, B_grad."""
        self._step_count += 1

        # Gradient cosine similarity
        cos_sim = 0.0
        with torch.no_grad():
            g_flat = lora_B_grad.detach().float().flatten()
            g_norm = g_flat / (g_flat.norm() + 1e-8)
            if domain == "code":
                self._mean_grad_code = (g_norm.clone() if self._mean_grad_code is None
                                        else 0.9 * self._mean_grad_code + 0.1 * g_norm)
            else:
                self._mean_grad_med  = (g_norm.clone() if self._mean_grad_med is None
                                        else 0.9 * self._mean_grad_med  + 0.1 * g_norm)

            if self._mean_grad_code is not None and self._mean_grad_med is not None:
                cos_sim = (self._mean_grad_code * self._mean_grad_med).sum().item()
                conflict = cos_sim < self.cosine_threshold
            else:
                conflict = False

        # Rank importance (single expert)
        with torch.no_grad():
            col_norms = lora_B.detach().float().norm(dim=0)
            row_norms = lora_A.detach().float().norm(dim=1)
            ri = (col_norms * row_norms).mean().item()
        self._ri_history.append(ri)

        if len(self._ri_history) < self.window:
            return False

        # Plateau check
        recent = self._ri_history[-self.window:]
        x = torch.arange(len(recent), dtype=torch.float32)
        y = torch.tensor(recent, dtype=torch.float32)
        slope = ((x*y).mean() - x.mean()*y.mean()) / (x.var(unbiased=False) + 1e-12)
        slope_val = slope.item()
        plateau = abs(slope_val) < self.tau_plateau

        if self._step_count % 50 == 0:
            print(f"    [monitor @{self._step_count}] "
                  f"ri_slope={slope_val:.2e} (need<{self.tau_plateau:.0e}) plateau={plateau} | "
                  f"cos_sim={cos_sim:.4f} (need<{self.cosine_threshold}) conflict={conflict}")

        self._plateau_window.append(plateau)
        self._conflict_window.append(conflict)
        if len(self._plateau_window) > self.window:
            self._plateau_window.pop(0)
            self._conflict_window.pop(0)

        if (len(self._plateau_window) == self.window
                and all(self._plateau_window)
                and all(self._conflict_window)):
            self._plateau_window.clear()
            self._conflict_window.clear()
            print(f"    [monitor @{self._step_count}] *** TRIGGER FIRED ***")
            return True
        return False

    def reset_after_spawn(self):
        self._ri_history.clear()
        self._plateau_window.clear()
        self._conflict_window.clear()

print("ConflictSaturationMonitor defined. (identical to OLMoE — cosine similarity version)")

In [ ]:
# DRLoRATracker — Tracks routing frequency and rank importance EMAs
# Identical to OLMoE implementation.

class DRLoRATracker:
    """
    Track routing frequency (Eq. 5) and rank importance (Eq. 6-8) with EMA.
    Uses forward hooks to capture actual router softmax weights.
    """
    def __init__(self, num_layers, num_experts_per_layer, ema_beta=0.9, device="cuda"):
        self.num_layers = num_layers
        self.num_experts_per_layer = num_experts_per_layer
        self.ema_beta = ema_beta
        self.device = device

        self.routing_frequency = torch.zeros(num_layers, num_experts_per_layer,
                                             device=device, dtype=torch.float32)
        self.rank_importance = torch.zeros(num_layers, num_experts_per_layer,
                                           device=device, dtype=torch.float32)
        self._routing_hooks = []
        self._current_router_weights: Dict[int, torch.Tensor] = {}
        self.num_updates = 0

    def _make_router_hook(self, layer_idx):
        def hook(module, input, output):
            if isinstance(output, tuple): logits = output[0]
            else: logits = output
            if logits.dim() == 3: logits = logits.view(-1, logits.shape[-1])
            probs = F.softmax(logits.float(), dim=-1)
            self._current_router_weights[layer_idx] = probs.detach()
        return hook

    def _remove_routing_hooks(self):
        for hook in self._routing_hooks: hook.remove()
        self._routing_hooks = []
        self._current_router_weights = {}

    def update_routing_frequency_from_hooks(self):
        for layer_idx, probs in self._current_router_weights.items():
            if layer_idx >= self.num_layers: continue
            mean_probs = probs.mean(dim=0)
            n_exp = min(mean_probs.shape[0], self.num_experts_per_layer)
            self.routing_frequency[layer_idx, :n_exp] = (
                self.ema_beta * self.routing_frequency[layer_idx, :n_exp] +
                (1 - self.ema_beta) * mean_probs[:n_exp].to(self.device)
            )
        self._current_router_weights = {}
        self.num_updates += 1

    def compute_rank_importance(self, lora_modules) -> Dict[Tuple[int, int], float]:
        importance = {}
        for info in lora_modules:
            layer_idx, expert_idx, dl = info["layer"], info["expert"], info["dr_lora"]
            if dl.lora_A.grad is None or dl.lora_B.grad is None:
                importance[(layer_idx, expert_idx)] = 0.0; continue
            active_mask = dl.rank_mask
            if not active_mask.any():
                importance[(layer_idx, expert_idx)] = 0.0; continue
            grad_A = dl.lora_A.grad[active_mask].abs().mean().item()
            grad_B = dl.lora_B.grad[:, active_mask].abs().mean().item()
            importance[(layer_idx, expert_idx)] = grad_A * grad_B
        return importance

    def update_rank_importance(self, importance: Dict[Tuple[int, int], float]):
        for (layer_idx, expert_idx), g_new in importance.items():
            if layer_idx < self.num_layers and expert_idx < self.num_experts_per_layer:
                self.rank_importance[layer_idx, expert_idx] = (
                    self.ema_beta * self.rank_importance[layer_idx, expert_idx] +
                    (1 - self.ema_beta) * g_new
                )

    def get_saliency(self, layer_idx, expert_idx, current_rank, gamma=0.5) -> float:
        f = self.routing_frequency[layer_idx, expert_idx].item()
        g = self.rank_importance[layer_idx, expert_idx].item()
        return (f * g) / ((current_rank + 1) ** gamma)

print("DRLoRATracker defined. (identical to OLMoE)")

In [ ]:
# DRLoRAGrowthSchedule — Manages rank growth timing and quotas
# Identical to OLMoE implementation.

class DRLoRAGrowthSchedule:
    def __init__(self, total_steps, warmup_steps, growth_interval,
                 r_init, r_target, r_max, num_experts_per_layer, num_layers,
                 end_buffer_steps=100):
        self.total_steps = total_steps
        self.warmup_steps = warmup_steps
        self.growth_interval = growth_interval
        self.r_init = r_init
        self.r_target = r_target
        self.r_max = r_max
        self.num_experts_per_layer = num_experts_per_layer
        self.num_layers = num_layers
        self.end_buffer_steps = end_buffer_steps

        effective_end = total_steps - end_buffer_steps
        growth_duration = max(1, effective_end - warmup_steps)
        self.num_growth_events = max(1, growth_duration // growth_interval)

        ranks_per_expert = r_target - r_init
        total_experts = num_experts_per_layer * num_layers
        self.total_ranks_to_add = ranks_per_expert * total_experts
        self.quota_per_event = max(1, self.total_ranks_to_add // self.num_growth_events)

        self.growth_event_steps = [
            warmup_steps + i * growth_interval
            for i in range(self.num_growth_events)
            if warmup_steps + i * growth_interval < effective_end
        ]
        self.ranks_per_expert_per_event = max(1, math.ceil(ranks_per_expert / self.num_growth_events))

        print(f"  DRLoRAGrowthSchedule: {self.num_growth_events} events at steps {self.growth_event_steps}")
        print(f"  ranks_per_expert_per_event={self.ranks_per_expert_per_event}")

    def can_grow_at_step(self, step): return step in self.growth_event_steps
    def get_event_index(self, step): return self.growth_event_steps.index(step) if step in self.growth_event_steps else -1
    def get_rank_quota(self, event_idx=None): return self.quota_per_event

print("DRLoRAGrowthSchedule defined. (identical to OLMoE)")

In [ ]:
# perform_rank_growth — Algorithm 1 lines 9-14
# Identical to OLMoE implementation.

def perform_rank_growth(tracker, schedule, lora_modules, current_step,
                        r_init, r_max, p_grow=0.5, gamma=0.5, verbose=False) -> dict:
    if not schedule.can_grow_at_step(current_step):
        return {"grew": False}

    n_new_per_expert = schedule.ranks_per_expert_per_event
    total_new_ranks = 0
    num_experts_grown = 0
    expert_growth = {}
    by_layer = defaultdict(list)
    for info in lora_modules:
        by_layer[info["layer"]].append(info)

    for layer_idx, experts in by_layer.items():
        saliencies = []
        for info in experts:
            dl = info["dr_lora"]
            current_rank = dl.get_active_ranks()
            sal = tracker.get_saliency(layer_idx, info["expert"], current_rank, gamma)
            saliencies.append((sal, info))
        saliencies.sort(key=lambda x: x[0], reverse=True)

        for sal, info in saliencies:
            dl = info["dr_lora"]
            current_rank = dl.get_active_ranks()
            free_slots = r_max - current_rank
            if free_slots <= 0: continue
            n_grow = min(n_new_per_expert, free_slots)
            for _ in range(n_grow):
                next_rank = dl.get_active_ranks()
                if next_rank < r_max:
                    dl.activate_rank(next_rank)
            actual_grown = dl.get_active_ranks() - current_rank
            if actual_grown > 0:
                expert_growth[(layer_idx, info["expert"])] = actual_grown
                total_new_ranks += actual_grown
                num_experts_grown += 1
                tracker.rank_importance[layer_idx, info["expert"]] = 0.0

    return {"grew": total_new_ranks > 0, "total_new_ranks": total_new_ranks,
            "num_experts_grown": num_experts_grown, "expert_growth": expert_growth}

print("perform_rank_growth defined. (identical to OLMoE)")

In [ ]:
# LoRAQwenExperts — Standard LoRA baseline for Qwen1.5-MoE
#
# Adapted from LoRAOLMoEExperts:
#   - Class renamed
#   - Shape: down_proj = [60, out_f=2048, in_f=1408]
#     → num_experts, out_f, in_f = shape  (transposed vs OLMoE)
#     → lora_dim = out_f = 2048  (same value as OLMoE's in_f)
#   - Shared expert NOT touched (handled by Qwen2MoeSparseMoeBlock)

class LoRAQwenExperts(nn.Module):
    """Standard LoRA wrapper for Qwen routing experts. Fixed-rank, no dynamic expansion."""

    def __init__(self, original_experts, base_rank=16, lora_alpha=32):
        super().__init__()
        self.original = original_experts
        for p in self.original.parameters():
            p.requires_grad = False

        # Qwen down_proj: [num_experts, out_f=model_dim, in_f=ffn_dim]
        # OLMoE down_proj: [num_experts, in_f=model_dim, out_f=ffn_dim]
        # Both → lora_dim = model_dim = 2048, but index differs
        self.num_experts, self.out_f, self.in_f = self.original.down_proj.shape
        self.dtype = self.original.down_proj.dtype
        self.lora_dim = self.out_f  # = 2048 = model_dim

        dev = self.original.down_proj.device

        self.base_loras = nn.ModuleList([
            HierarchicalExpert(
                in_features=self.lora_dim,
                out_features=self.lora_dim,
                base_rank=base_rank,
                lora_alpha=lora_alpha,
            ).to(dev).to(self.dtype)
            for _ in range(self.num_experts)
        ])

        print(f"LoRAQwenExperts: down_proj={self.original.down_proj.shape}")
        print(f"  → lora_dim={self.lora_dim} (out_f={self.out_f}, in_f={self.in_f}) ✓")

    @property
    def device(self): return self.original.down_proj.device

    def forward(self, hidden_states, router_top_k_indices, router_top_k_weights):
        orig_out = self.original(hidden_states, router_top_k_indices, router_top_k_weights)
        correction = torch.zeros_like(orig_out)
        for k in range(self.num_experts):
            mask = (router_top_k_indices == k)
            if not mask.any(): continue
            eff_w = (router_top_k_weights * mask.to(router_top_k_weights.dtype)).sum(dim=1)
            token_mask = eff_w > 0
            if not token_mask.any(): continue
            x_k = hidden_states[token_mask].to(orig_out.dtype)
            lora_out = self.base_loras[k](x_k)
            correction[token_mask] += eff_w[token_mask].to(orig_out.dtype).unsqueeze(-1) * lora_out
        return orig_out + correction

print("LoRAQwenExperts defined.")

In [ ]:
# DRLoRAQwenExperts — DR-LoRA with saliency-based rank growth for Qwen1.5-MoE
#
# Adapted from DRLoRAOLMoEExperts:
#   - Class renamed
#   - Shape: num_experts, out_f, in_f = down_proj.shape → in_features=out_features=out_f=2048
#   - Same forward loop as LoRAQwenExperts

class DRLoRAQwenExperts(nn.Module):
    def __init__(self, original_experts, r_max=16, r_init=4, lora_alpha=16):
        super().__init__()
        self.original = original_experts
        for p in self.original.parameters():
            p.requires_grad = False

        self.num_experts, self.out_f, self.in_f = self.original.down_proj.shape
        self.dtype = self.original.down_proj.dtype
        self.r_max = r_max
        self.r_init = r_init
        self.lora_dim = self.out_f  # 2048

        dev = self.original.down_proj.device

        self.dr_loras = nn.ModuleList([
            DRLoRALayer(
                in_features=self.out_f,
                out_features=self.out_f,
                r_max=r_max, r_init=r_init, lora_alpha=lora_alpha, lora_dropout=0.0,
            ).to(device=dev, dtype=self.dtype)
            for _ in range(self.num_experts)
        ])
        with torch.no_grad():
            for dl in self.dr_loras:
                dl.lora_B.zero_()

        print(f"DRLoRAQwenExperts: out_f={self.out_f}, r_init={r_init}, r_max={r_max} ✓")

    @property
    def device(self): return self.original.down_proj.device

    def get_lora_modules_list(self, layer_idx):
        return [{"layer": layer_idx, "expert": k, "dr_lora": self.dr_loras[k], "module": "packed_lora"}
                for k in range(self.num_experts)]

    def forward(self, hidden_states, router_top_k_indices, router_top_k_weights):
        orig_out = self.original(hidden_states, router_top_k_indices, router_top_k_weights)
        correction = torch.zeros_like(orig_out)
        for k in range(self.num_experts):
            mask = (router_top_k_indices == k)
            if not mask.any(): continue
            eff_w = (router_top_k_weights * mask.to(router_top_k_weights.dtype)).sum(dim=1)
            active = eff_w.abs() > 1e-6
            if not active.any(): continue
            x_k = hidden_states[active].to(orig_out.dtype)
            lora_out = self.dr_loras[k](x_k)
            correction[active] += eff_w[active].to(orig_out.dtype).unsqueeze(1) * lora_out
        return orig_out + correction

print("DRLoRAQwenExperts defined.")

In [ ]:
# HierarchicalQwenExperts — ReLU-gated sub-adapter spawning for Qwen1.5-MoE
# Robust version: nn.ModuleList + register_parameter (for checkpoint serialization).
#
# Adapted from HierarchicalOLMoEExperts:
#   - Class renamed
#   - Shape: num_experts, out_f, in_f = down_proj.shape → lora_dim = out_f = 2048
#   - Shared expert (mlp.shared_expert) is NOT wrapped — Qwen2MoeSparseMoeBlock
#     adds it independently after our wrapper returns
#   - AlignDevicesHook not needed (single A100)
#
# What is identical to OLMoE:
#   - HierarchicalExpert sub-adapter (B=0 zero-spawn guarantee)
#   - spawn() with SVD init and gate noise formula
#   - forward() loop structure
#   - rebuild_gate_registry() for checkpoint restore

class HierarchicalQwenExperts(nn.Module):
    """
    Hierarchical ReLU-LoRA for Qwen1.5-MoE routing experts.
    E_k(x) = W_k x + L_{k,0}(x) + sum_j ReLU(w_{k,j}^T x) * L_{k,j}(x)

    Only wraps mlp.experts (routing experts). mlp.shared_expert fires separately
    inside Qwen2MoeSparseMoeBlock and is fully transparent to this wrapper.

    down_proj shape: [60, out_f=2048, in_f=1408]
    lora_dim = out_f = 2048  (same as OLMoE's in_f=2048 — same value, different index)
    """

    def __init__(self, original_experts, base_rank=16, lora_alpha=32):
        super().__init__()
        self.original = original_experts
        for p in self.original.parameters():
            p.requires_grad = False

        # Qwen: [num_experts, out_f=model_dim, in_f=ffn_dim]
        self.num_experts, self.out_f, self.in_f = self.original.down_proj.shape
        self.dtype = self.original.down_proj.dtype
        self.lora_dim = self.out_f  # = 2048 = model_dim  (NOT in_f=1408)

        dev = self.original.down_proj.device

        self.base_loras = nn.ModuleList([
            HierarchicalExpert(
                in_features=self.lora_dim,
                out_features=self.lora_dim,
                base_rank=base_rank,
                lora_alpha=lora_alpha,
            ).to(dev).to(self.dtype)
            for _ in range(self.num_experts)
        ])

        # nn.ModuleList for proper PyTorch tracking and checkpoint serialization
        self.spawn_loras = nn.ModuleList([
            nn.ModuleList() for _ in range(self.num_experts)
        ])
        self._spawn_gate_names: dict = {}  # (expert_id, spawn_idx) -> param name

        print(f"HierarchicalQwenExperts: lora_dim={self.lora_dim} (out_f={self.out_f}) ✓")

    @property
    def device(self): return self.original.down_proj.device

    def spawn(self, expert_id, rank=8, weight_grad=None) -> list:
        """Spawn a ReLU-gated sub-adapter. Returns new parameters for optimizer."""
        dev, dtype = self.device, self.dtype

        lora = HierarchicalExpert(
            in_features=self.lora_dim,
            out_features=self.lora_dim,
            base_rank=rank,
            lora_alpha=2 * rank,
        ).to(dev).to(dtype)

        if weight_grad is not None:
            try:
                U, S, Vh = torch.linalg.svd(weight_grad.float(), full_matrices=False)
                with torch.no_grad():
                    lora.A.copy_(Vh[:rank].to(dtype))
                print(f"    [spawn] Expert {expert_id}: SVD init, top-sv={S[0].item():.4f}")
            except Exception as e:
                print(f"    [spawn] Expert {expert_id}: SVD failed ({e}), random init")

        # Gate noise = 1e-3 * var(W_k), matching Phi-MoE proposal §3.3
        sigma = 1e-3 * self.original.down_proj[expert_id].float().var().item()
        gate_val = torch.randn(self.lora_dim, device=dev, dtype=dtype) * sigma

        spawn_idx = len(self.spawn_loras[expert_id])
        gate_name = f"spawn_gate_{expert_id}_{spawn_idx}"
        gate = nn.Parameter(gate_val)
        self.register_parameter(gate_name, gate)
        self._spawn_gate_names[(expert_id, spawn_idx)] = gate_name

        self.spawn_loras[expert_id].append(lora)
        return list(lora.parameters()) + [gate]

    def forward(self, hidden_states, router_top_k_indices, router_top_k_weights):
        orig_out = self.original(hidden_states, router_top_k_indices, router_top_k_weights)
        correction = torch.zeros_like(orig_out)

        for k in range(self.num_experts):
            mask = (router_top_k_indices == k)
            if not mask.any(): continue
            eff_w = (router_top_k_weights * mask.to(router_top_k_weights.dtype)).sum(dim=1)
            token_mask = eff_w > 0
            if not token_mask.any(): continue
            x_k = hidden_states[token_mask].to(orig_out.dtype)
            w_k = eff_w[token_mask].to(orig_out.dtype)

            base_out = self.base_loras[k](x_k)
            correction[token_mask] += w_k.unsqueeze(-1) * base_out

            for spawn_idx, sub_lora in enumerate(self.spawn_loras[k]):
                gate_name = self._spawn_gate_names.get((k, spawn_idx))
                if gate_name is None: continue
                gate_vec = getattr(self, gate_name)
                g = F.relu(x_k @ gate_vec.to(orig_out.dtype))
                sub_out = sub_lora(x_k)
                correction[token_mask] += (g * w_k).unsqueeze(-1) * sub_out

        return orig_out + correction

    def rebuild_gate_registry(self):
        """Call after load_state_dict to reconnect _spawn_gate_names from registered params."""
        self._spawn_gate_names = {}
        for expert_id in range(self.num_experts):
            for spawn_idx in range(len(self.spawn_loras[expert_id])):
                gate_name = f"spawn_gate_{expert_id}_{spawn_idx}"
                if hasattr(self, gate_name):
                    self._spawn_gate_names[(expert_id, spawn_idx)] = gate_name

print("HierarchicalQwenExperts defined.")

## Section 3 — Method Factory


In [ ]:
# Method Factory — setup_method returns (wrapper, optimizer, extras)
# Adapted from OLMoE: uses Qwen wrappers, TARGET_LAYERS=[0], no AlignDevicesHook.

class DRLoRAWrapper:
    def __init__(self, packed_wrapper, tracker, schedule, layer_idx):
        self.packed_wrapper = packed_wrapper
        self.tracker = tracker
        self.schedule = schedule
        self.layer_idx = layer_idx
        self.lora_modules = packed_wrapper.get_lora_modules_list(layer_idx)
        self.expanded = False


def get_clean_original_experts(model, target_layer):
    current = model.model.layers[target_layer].mlp.experts
    while hasattr(current, 'original'):
        current = current.original
    return current


TARGET_LAYERS = [0]  # Layer 0 — highest top-K Jaccard for Qwen (= 0.154)

def setup_method(model, method_name: str, cfg: dict):
    gc.collect()
    torch.cuda.empty_cache()
    base_rank = cfg["base_rank"]
    lr = cfg["lr"]
    n_steps = cfg["n_steps"]

    for p in model.parameters():
        p.requires_grad = False

    wrappers = []

    if method_name == "lora":
        for target_layer in TARGET_LAYERS:
            original_experts = get_clean_original_experts(model, target_layer)
            wrapper = LoRAQwenExperts(original_experts, base_rank, base_rank * 2)
            model.model.layers[target_layer].mlp.experts = wrapper
            wrappers.append(wrapper)
        all_params = [p for w in wrappers for p in w.parameters() if p.requires_grad]
        optimizer = torch.optim.AdamW(all_params, lr=lr)
        return wrappers[0], optimizer, {"monitor": None, "all_wrappers": wrappers}

    elif method_name == "hierarchical":
        for target_layer in TARGET_LAYERS:
            # Confirm shared expert is present (Qwen-specific)
            assert hasattr(model.model.layers[target_layer].mlp, 'shared_expert'), \
                "shared_expert not found — check Qwen model structure"
            original_experts = get_clean_original_experts(model, target_layer)
            wrapper = HierarchicalQwenExperts(original_experts, base_rank * 2, base_rank * 4)
            model.model.layers[target_layer].mlp.experts = wrapper
            wrappers.append(wrapper)
        monitor = ConflictSaturationMonitor(
            tau_plateau=cfg.get("tau_plateau", 1e-4),
            delta_threshold=cfg.get("delta_threshold", 1.0),
            window=cfg.get("window", 15),
        )
        all_params = [p for w in wrappers for p in w.parameters() if p.requires_grad]
        optimizer = torch.optim.AdamW(all_params, lr=lr)
        return wrappers[0], optimizer, {"monitor": monitor, "all_wrappers": wrappers}

    elif method_name == "dr_lora":
        target_layer = cfg["target_layer"]
        original_experts = get_clean_original_experts(model, target_layer)
        r_init = max(2, base_rank // 2)
        r_max  = base_rank * 2
        packed_wrapper = DRLoRAQwenExperts(original_experts, r_max, r_init, base_rank)
        model.model.layers[target_layer].mlp.experts = packed_wrapper

        # No AlignDevicesHook needed for Qwen on single A100

        warmup  = max(10, n_steps // 10)
        interval = max(50, n_steps // 6)
        end_buf  = max(50, n_steps // 8)
        schedule = DRLoRAGrowthSchedule(
            total_steps=n_steps, warmup_steps=warmup, growth_interval=interval,
            r_init=r_init, r_target=r_max, r_max=r_max,
            num_experts_per_layer=packed_wrapper.num_experts, num_layers=1,
            end_buffer_steps=end_buf,
        )
        tracker = DRLoRATracker(
            num_layers=model.config.num_hidden_layers,
            num_experts_per_layer=packed_wrapper.num_experts,
            ema_beta=0.9, device="cuda" if torch.cuda.is_available() else "cpu",
        )
        # Hook Qwen's router gate — Qwen2MoeSparseMoeBlock uses attribute 'gate'
        for i, hook_layer in enumerate(model.model.layers):
            if i != target_layer:
                continue
            gate = getattr(hook_layer.mlp, "gate", None)
            if gate is None:
                # Fallback: search common router attribute names
                for attr in ["router", "moe_gate", "expert_gate"]:
                    gate = getattr(hook_layer.mlp, attr, None)
                    if gate is not None: break
            if gate is not None:
                hook = gate.register_forward_hook(tracker._make_router_hook(target_layer))
                tracker._routing_hooks.append(hook)
                print(f"  DR-LoRA: hooked router at layer {target_layer} (attr: {type(gate).__name__})")
            else:
                print(f"  DR-LoRA: ⚠ router not found at layer {target_layer} — frequency tracking disabled")

        optimizer = torch.optim.AdamW(
            [p for p in packed_wrapper.parameters() if p.requires_grad], lr=lr)
        dr_wrapper = DRLoRAWrapper(packed_wrapper, tracker, schedule, target_layer)
        return dr_wrapper, optimizer, {"r_init": r_init, "r_max": r_max}

    else:
        raise ValueError(f"Unknown method: {method_name}")

print("Method factory ready (lora / hierarchical / dr_lora)")

## Section 4 — Configuration & Evaluation


In [ ]:
# Experiment configuration
# Identical to OLMoE except target_layer: 8 → 0

COMPARISON_CFG = {
    "methods": [#"lora",
                #"dr_lora",
                "hierarchical"],
    "target_layer": 0,   # Layer 0 — highest Jaccard for Qwen Python vs Medical
    "base_rank": 8,
    "lr": 5e-5,
    "n_steps": 1500,
    "batch_size": 1,
    "max_length": 256,
    "log_every": 100,
    "eval_every": 150,

    # Spawn trigger parameters — calibrated for Qwen loss scale
    # Note: main grid uses fixed-step spawning (not dynamic trigger)
    # These are used only in the Phase 2 calibration cell.
    "delta_threshold": 0.5,   # |ema divergence| threshold (nats)
    "tau_plateau": 1e-04,      # RI slope flatline threshold
    "window": 15,              # consecutive steps both conditions must hold
    "max_sub_adapters": 10,    # cap on spawns per expert

    # Conflict-scaling grid — identical to OLMoE
    "conflict_ratios": {
        "A": 0.0,   # 100% code (no conflict baseline)
        "B": 0.2,   # 80/20 code/medical (moderate conflict)
        "C": 0.5,   # 50/50 (severe conflict)
    },

    "p_grow": 0.1,
    "gamma": 0.5,
}

print("Configuration:")
for k, v in COMPARISON_CFG.items():
    print(f"  {k}: {v}")

In [ ]:
# ── TEST MODE SWITCH ──────────────────────────────────────────────────────────
#
# TEST_MODE = True  →  tiny dataset + minimal steps to verify the pipeline runs
#                       without crashes. Results are NOT meaningful.
#
# TEST_MODE = False →  full experiment (1000 samples, 1500 steps per run).
#                       Use this for thesis results on Colab A100.

TEST_MODE = True   # <-- toggle here

if TEST_MODE:
    print("=" * 60)
    print("  TEST_MODE = True")
    print("  Pipeline check only — results are NOT meaningful.")
    print("  Set TEST_MODE = False for real experiments.")
    print("=" * 60)
    # Shrink training budget
    COMPARISON_CFG["n_steps"]    = 20
    COMPARISON_CFG["log_every"]  = 5
    COMPARISON_CFG["eval_every"] = 10
    # Shrink HumanEval eval texts (used as :_N_EVAL in training loop)
    _N_EVAL = 2
else:
    print("TEST_MODE = False — full experiment configuration active.")
    print(f"  n_steps={COMPARISON_CFG['n_steps']}, N_TOTAL=1000 (set in dataset cell)")
    _N_EVAL = 20

In [ ]:
# ── Jaccard Routing Overlap Diagnostic for Qwen1.5-MoE ───────────────────────
# Qwen: 60 routing experts, top-4, 24 layers.
# Uses Top-K Jaccard (TOP_N=15 ≈ top quartile of 60) — required because raw
# Jaccard ≈ 1.0 (with 4/60 routing, even 40 examples cover most experts).
# Pre-run result: Layer 0, Jaccard=0.154 (Python vs Medical).
# This cell re-confirms the target and identifies the highest-conflict expert.

from collections import defaultdict as _dd
import torch

TOP_K_QWEN         = 4    # Qwen uses top-4 routing
TOP_N              = 15   # top-quartile of 60 experts (25%)
N_JACCARD_EXAMPLES = 5 if TEST_MODE else 40
JACCARD_MAX_LEN    = 128

def _get_expert_topn_freq_qwen(model, tokenizer, texts, top_k=4, n=40, max_len=128, top_n=15):
    """
    Collect per-layer top-N expert sets and frequency counts.
    Returns: per_layer_sets (domain sets), per_layer_freq (routing freq dicts)
    """
    freq = _dd(lambda: _dd(int))  # freq[layer][expert] = count
    total_per_layer = _dd(int)
    model.eval()
    with torch.no_grad():
        for text in texts[:n]:
            enc = tokenizer(text, return_tensors='pt', truncation=True,
                            max_length=max_len, padding=False).to(model.device)
            out = model(**enc, output_router_logits=True, return_dict=True)
            if out.router_logits is None: continue
            for layer_idx, logits in enumerate(out.router_logits):
                if logits is None: continue
                if logits.dim() == 3: logits = logits.view(-1, logits.shape[-1])
                top_exp = logits.topk(top_k, dim=-1).indices
                for eid in top_exp.flatten().tolist():
                    freq[layer_idx][eid] += 1
                    total_per_layer[layer_idx] += 1
    model.train()
    # Normalize to frequency
    result = {}
    for layer_idx, ecounts in freq.items():
        total = max(total_per_layer[layer_idx], 1)
        result[layer_idx] = {e: c / total for e, c in ecounts.items()}
    return result

def _topk_jaccard(freq_a, freq_b, top_n=15):
    """Per-layer top-N Jaccard: intersection/union of top-N most-routed experts."""
    all_layers = sorted(set(freq_a) | set(freq_b))
    per_layer = {}
    for l in all_layers:
        pa, pb = freq_a.get(l, {}), freq_b.get(l, {})
        top_a = set(sorted(pa, key=pa.get, reverse=True)[:top_n])
        top_b = set(sorted(pb, key=pb.get, reverse=True)[:top_n])
        shared = top_a & top_b
        union  = top_a | top_b
        per_layer[l] = len(shared) / len(union) if union else 0.0
    return per_layer

print("Loading probe texts for Qwen Jaccard diagnostic...")
from datasets import load_dataset as _lds

_he = _lds("openai/openai_humaneval", split="test")
_pq = _lds("qiaojin/PubMedQA", "pqa_labeled", split="train")
_code_texts    = [str(ex["prompt"])   for ex in _he.select(range(min(N_JACCARD_EXAMPLES, len(_he))))]
_medical_texts = [str(ex["question"]) for ex in _pq.select(range(min(N_JACCARD_EXAMPLES, len(_pq))))]

print(f"Loaded {len(_code_texts)} code / {len(_medical_texts)} medical probe texts.")
print("Computing routing frequencies for Qwen (this takes ~2–3 min)...")

# Use a clean model — reload fresh
_jac_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, torch_dtype=torch.bfloat16, device_map="cuda:0", trust_remote_code=True)
_jac_model.config.use_cache = True

freq_code    = _get_expert_topn_freq_qwen(_jac_model, tokenizer, _code_texts,
                                           top_k=TOP_K_QWEN, n=N_JACCARD_EXAMPLES, top_n=TOP_N)
freq_medical = _get_expert_topn_freq_qwen(_jac_model, tokenizer, _medical_texts,
                                           top_k=TOP_K_QWEN, n=N_JACCARD_EXAMPLES, top_n=TOP_N)

del _jac_model; gc.collect(); torch.cuda.empty_cache()

per_layer_j = _topk_jaccard(freq_code, freq_medical, top_n=TOP_N)
sorted_layers = sorted(per_layer_j.items(), key=lambda x: x[1], reverse=True)

print("\n" + "=" * 60)
print(f"JACCARD PER-LAYER OVERLAP  (Qwen1.5-MoE, top-{TOP_N} of 60)")
print(f"{'Layer':>6}  {'Jaccard':>8}  bar")
print("-" * 60)
for layer, score in sorted_layers:
    bar = "\u2588" * int(score * 40)
    marker = " \u25c4 TARGET" if layer == COMPARISON_CFG["target_layer"] else ""
    print(f"  {layer:>4}   {score:.4f}   {bar}{marker}")

best_layer = sorted_layers[0][0]
best_score = sorted_layers[0][1]
print("=" * 60)
print(f"\nHighest overlap: Layer {best_layer}  (Jaccard={best_score:.4f})")

# Identify highest-conflict expert in target layer
TARGET_EXPERT_DIAG = 5  # from pre-run Jaccard diagnostic
print(f"\nPre-run result: TARGET_LAYER=0, TARGET_EXPERT=5")
print(f"  Freq_python=0.0230, Freq_medical=0.0242, conflict_score=0.0236")
if best_layer == COMPARISON_CFG["target_layer"]:
    print(f"\u2705 Layer {best_layer} confirmed as highest-overlap target.")
else:
    print(f"\u26a0  Best layer is {best_layer}, but using pre-validated layer 0.")
    print(f"   Update TARGET_LAYERS and COMPARISON_CFG['target_layer'] if needed.")

In [ ]:
# Multi-Domain Dataset Builder — identical to OLMoE
#
# FIX: All runs use same total budget (N_TOTAL=1000). Only split changes.
#   Run A: 1000 code + 0 medical   (TEST_MODE: 20 + 0)
#   Run B: 800 code + 200 medical  (TEST_MODE: 16 + 4)
#   Run C: 500 code + 500 medical  (TEST_MODE: 10 + 10)
#
# Code: bigcode/the-stack-smol (real Python files)
# Medical: qiaojin/PubMedQA

from datasets import load_dataset

N_TOTAL = 20 if TEST_MODE else 1000  # TEST_MODE shrinks to 20 for pipeline check

def load_code_texts(n):
    print("  Loading code: bigcode/the-stack-smol (Python)...")
    ds = load_dataset("bigcode/the-stack-smol", data_dir="data/python",
                      split="train", streaming=True)
    texts = []
    for ex in ds:
        content = ex.get("content", "")
        if len(content) > 100 and len(content) < 4000:
            texts.append(content[:2000])
        if len(texts) >= n:
            break
    print(f"    Loaded {len(texts)} Python code samples")
    return texts

def load_medical_texts(n):
    print("  Loading medical: qiaojin/PubMedQA...")
    ds = load_dataset("qiaojin/PubMedQA", "pqa_labeled", split="train")
    texts = [str(ex["question"]) for ex in ds][:n]
    print(f"    Loaded {len(texts)} medical samples")
    return texts

_CODE_CACHE = None
_MED_CACHE  = None

def build_conflict_dataloader(tokenizer, conflict_ratio=0.5, max_length=256):
    """Build balanced interleaved Code+Medical dataset. N_TOTAL fixed across all runs."""
    global _CODE_CACHE, _MED_CACHE

    n_medical = int(N_TOTAL * conflict_ratio)
    n_code    = N_TOTAL - n_medical

    print(f"\nBuilding dataset: {n_code} code + {n_medical} medical = {N_TOTAL} total")

    if _CODE_CACHE is None or len(_CODE_CACHE) < N_TOTAL:
        _CODE_CACHE = load_code_texts(N_TOTAL)
    if n_medical > 0 and (_MED_CACHE is None or len(_MED_CACHE) < n_medical):
        _MED_CACHE = load_medical_texts(N_TOTAL)

    code_texts = _CODE_CACHE[:n_code]
    med_texts  = (_MED_CACHE[:n_medical] if n_medical > 0 else [])

    texts, domains = [], []
    if n_medical == 0:
        texts, domains = list(code_texts), ["code"] * len(code_texts)
    elif n_code == 0:
        texts, domains = list(med_texts), ["medical"] * len(med_texts)
    else:
        ratio = n_medical / n_code
        ci = pi = 0
        acc = 0.0
        while pi < len(code_texts) or ci < len(med_texts):
            if pi < len(code_texts):
                texts.append(code_texts[pi]); domains.append("code"); pi += 1
            acc += ratio
            while acc >= 1.0 and ci < len(med_texts):
                texts.append(med_texts[ci]); domains.append("medical"); ci += 1
                acc -= 1.0

    print(f"  Final: {domains.count('code')} code / {domains.count('medical')} medical")

    encodings = tokenizer(texts, padding='max_length', truncation=True,
                          max_length=max_length, return_tensors='pt')
    return encodings, domains


# Eval probe: hardcoded Python snippets (not HumanEval — avoid contamination)
EVAL_CODE_TEXTS = [
    "def binary_search(arr, target):\n    lo, hi = 0, len(arr)-1\n    while lo <= hi:\n        mid = (lo+hi)//2\n        if arr[mid] == target: return mid\n        elif arr[mid] < target: lo = mid+1\n        else: hi = mid-1\n    return -1",
    "def merge_sort(arr):\n    if len(arr) <= 1: return arr\n    mid = len(arr)//2\n    left = merge_sort(arr[:mid])\n    right = merge_sort(arr[mid:])\n    result = []\n    i = j = 0\n    while i < len(left) and j < len(right):\n        if left[i] <= right[j]: result.append(left[i]); i+=1\n        else: result.append(right[j]); j+=1\n    result.extend(left[i:]); result.extend(right[j:])\n    return result",
    "class LinkedList:\n    def __init__(self):\n        self.head = None\n    def append(self, val):\n        node = Node(val)\n        if not self.head:\n            self.head = node; return\n        cur = self.head\n        while cur.next: cur = cur.next\n        cur.next = node",
    "def fibonacci(n):\n    if n <= 1: return n\n    a, b = 0, 1\n    for _ in range(2, n+1): a, b = b, a+b\n    return b",
    "def is_prime(n):\n    if n < 2: return False\n    for i in range(2, int(n**0.5)+1):\n        if n % i == 0: return False\n    return True",
    "import re\ndef extract_emails(text):\n    pattern = r'[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\\.[a-zA-Z]{2,}'\n    return re.findall(pattern, text)",
    "def flatten(lst):\n    result = []\n    for item in lst:\n        if isinstance(item, list): result.extend(flatten(item))\n        else: result.append(item)\n    return result",
    "def lru_cache_impl(capacity):\n    from collections import OrderedDict\n    cache = OrderedDict()\n    def get(key):\n        if key not in cache: return -1\n        cache.move_to_end(key); return cache[key]\n    def put(key, value):\n        if key in cache: cache.move_to_end(key)\n        cache[key] = value\n        if len(cache) > capacity: cache.popitem(last=False)\n    return get, put",
] * 6  # 48 examples for stable PPL estimate

print("Dataset builder ready. N_TOTAL =", N_TOTAL)

In [ ]:
# ── Phase 2: Trigger Calibration Grid ────────────────────────────────────────
# Find (delta_threshold, tau_plateau) giving 1–3 spawns per 100 steps on Qwen.
# Run before main grid; update COMPARISON_CFG with locked values.
# Note: main grid uses fixed-step spawning — this cell validates the trigger
# parameters for informational purposes and thesis documentation.

import itertools, gc

_CALIB_TARGET_EXPERT = 5   # Qwen target expert
DELTA_VALUES         = [0.5] if TEST_MODE else [0.3, 0.5, 1.0]
TAU_PLATEAU_VALUES   = [1e-4] if TEST_MODE else [1e-3, 5e-4, 1e-4]
CALIB_STEPS          = 20 if TEST_MODE else 300

def run_calibration_trial(delta_threshold, tau_plateau, n_steps=CALIB_STEPS):
    gc.collect(); torch.cuda.empty_cache()

    _model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID, torch_dtype=torch.bfloat16, device_map="cuda:0", trust_remote_code=True)
    _model.gradient_checkpointing_disable()
    _model.config.use_cache = False

    cfg_calib = {**COMPARISON_CFG,
                 "delta_threshold": delta_threshold,
                 "tau_plateau": tau_plateau}
    _wrapper, _opt, _extras = setup_method(_model, "hierarchical", cfg_calib)
    _monitor = _extras["monitor"]

    enc, doms = build_conflict_dataloader(tokenizer, conflict_ratio=0.5,
                                          max_length=COMPARISON_CFG["max_length"])
    n_samp = enc["input_ids"].shape[0]
    spawn_steps = []

    _model.train()
    for step in range(n_steps):
        idx    = step % n_samp
        bi     = enc["input_ids"][idx:idx+1].to(_model.device)
        bm     = enc["attention_mask"][idx:idx+1].to(_model.device)
        domain = doms[idx]
        labels = bi.clone(); labels[:, :-1] = bi[:, 1:]; labels[:, -1] = -100

        _opt.zero_grad()
        loss = _model(input_ids=bi, attention_mask=bm, labels=labels).loss
        loss.backward()

        lora = _wrapper.base_loras[_CALIB_TARGET_EXPERT]
        if lora.A.grad is not None and lora.B.grad is not None:
            # Pass lora_B_grad explicitly — required by OLMoE-version monitor signature
            trigger = _monitor.update(lora.A.data, lora.B.data,
                                      lora.B.grad, loss.item(), domain)
            if trigger and len(_wrapper.spawn_loras[_CALIB_TARGET_EXPERT]) < 10:
                wg = lora.B.grad.float() @ lora.A.grad.float()
                new_p = _wrapper.spawn(_CALIB_TARGET_EXPERT, rank=8, weight_grad=wg)
                _opt.add_param_group({'params': new_p, 'lr': cfg_calib["lr"]})
                _monitor.reset_after_spawn()
                spawn_steps.append(step)
        _opt.step()

    del _model; gc.collect(); torch.cuda.empty_cache()
    spawn_rate = len(spawn_steps) / (n_steps / 100)
    return {"delta": delta_threshold, "tau": tau_plateau,
            "spawns": len(spawn_steps), "rate": spawn_rate, "steps": spawn_steps}

print("=" * 65)
print("PHASE 2: TRIGGER CALIBRATION GRID  (Qwen1.5-MoE)")
print(f"Grid: delta={DELTA_VALUES} x tau_plateau={TAU_PLATEAU_VALUES}")
print(f"Each trial: {CALIB_STEPS} steps, 50% conflict, Expert {_CALIB_TARGET_EXPERT}")
print("=" * 65)

calib_results = []
for delta, tau in itertools.product(DELTA_VALUES, TAU_PLATEAU_VALUES):
    print(f"\n-- delta={delta}  tau_plateau={tau:.0e} --")
    r = run_calibration_trial(delta, tau)
    calib_results.append(r)
    tag = ("\u2705 GOOD" if 1 <= r["rate"] <= 3
           else "\U0001F525 TOO MANY" if r["rate"] > 3 else "\u274c NONE")
    print(f"   Spawns: {r['spawns']}  Rate: {r['rate']:.1f}/100 steps  {tag}")

print("\n" + "=" * 65)
print("SUMMARY")
print(f"{'delta':>8}  {'tau_plateau':>12}  {'spawns':>7}  {'rate/100':>9}  verdict")
print("-" * 65)
good = []
for r in calib_results:
    tag = "\u2705 GOOD" if 1 <= r["rate"] <= 3 else ("\U0001F525 TOO MANY" if r["rate"] > 3 else "\u274c NONE")
    print(f"{r['delta']:>8.1f}  {r['tau']:>12.0e}  {r['spawns']:>7}  {r['rate']:>9.1f}  {tag}")
    if 1 <= r["rate"] <= 3:
        good.append(r)
print("=" * 65)
if good:
    best = min(good, key=lambda r: r["rate"])
    print(f"\n\u2705 Confirmed hyperparameters:")
    print(f"   delta_threshold = {best['delta']}")
    print(f"   tau_plateau     = {best['tau']:.0e}")
    COMPARISON_CFG["delta_threshold"] = best["delta"]
    COMPARISON_CFG["tau_plateau"]     = best["tau"]
    print("   (COMPARISON_CFG updated in-place)")
else:
    print("\n\u26a0  No setting produced spawns. Try lowering delta_threshold to 0.3.")

In [ ]:
# Perplexity evaluation — identical to OLMoE

def evaluate_perplexity(model, tokenizer, texts, max_length=256):
    model.eval()
    model.config.use_cache = True   # always on during eval
    total_loss = 0
    total_tokens = 0
    with torch.no_grad():
        for text in texts:
            enc = tokenizer(text, return_tensors='pt', truncation=True,
                            max_length=max_length).to(model.device)
            labels = enc['input_ids'].clone()
            labels[:, :-1] = enc['input_ids'][:, 1:]
            labels[:, -1] = -100
            outputs = model(input_ids=enc['input_ids'],
                            attention_mask=enc['attention_mask'], labels=labels)
            n_tokens = (labels != -100).sum().item()
            total_loss += outputs.loss.item() * n_tokens
            total_tokens += n_tokens
    model.train()
    model.config.use_cache = False  # back to training mode
    return math.exp(total_loss / max(total_tokens, 1))

print("evaluate_perplexity defined. (identical to OLMoE)")

## Section 5 — Comparison Training Grid


In [ ]:
# Main Comparison Training Loop — Conflict-Scaling Grid (Qwen1.5-MoE)
#
# Identical to OLMoE except:
#   - Model: Qwen/Qwen1.5-MoE-A2.7B (trust_remote_code, gradient_checkpointing_disable)
#   - TARGET_EXPERT = 5 (vs 0 in OLMoE)
#   - Grad debug: checks 'layers.0' (not 'layers.8')
#   - Checkpoint filenames: qwen_{run_key}.pt
#
# Fixed-step spawning: [150, 400, 650, 900, 1100] under conflict (bypasses noisy trigger)
# JSD probe: logged at each eval checkpoint for hierarchical runs

import math, numpy as np
from collections import defaultdict

# JSD helpers (identical to OLMoE)

def _collect_gate_activations_qwen(wrapper, texts, tokenizer, model,
                                    target_layer, target_expert,
                                    max_length=128, n_examples=30):
    """Collect ReLU gate activation values for each spawned sub-adapter."""
    activations = defaultdict(list)
    model.eval()
    with torch.no_grad():
        for text in texts[:n_examples]:
            inputs = tokenizer(text, return_tensors='pt', truncation=True,
                               max_length=max_length, padding=False).to(model.device)
            captured = {}
            def _hook(module, inp, out):
                h = inp[0].detach()
                if h.dim() == 3: h = h.view(-1, h.shape[-1])
                captured['h'] = h
            hook = model.model.layers[target_layer].mlp.register_forward_hook(_hook)
            model(**inputs)
            hook.remove()
            if 'h' not in captured: continue
            hidden = captured['h']
            for j, sub_lora in enumerate(wrapper.spawn_loras[target_expert]):
                gate_name = wrapper._spawn_gate_names.get((target_expert, j))
                if gate_name is None: continue
                gate_vec = getattr(wrapper, gate_name)
                acts = F.relu(hidden @ gate_vec.to(hidden.dtype)).float().cpu().tolist()
                activations[j].extend(acts)
    model.train()
    return activations

def _compute_jsd_scalar(p_vals, q_vals, n_bins=40):
    if len(p_vals) < 5 or len(q_vals) < 5: return float('nan')
    all_v = p_vals + q_vals
    lo, hi = min(all_v), max(all_v)
    if hi - lo < 1e-8: return 0.0
    bins = np.linspace(lo, hi, n_bins + 1)
    p_h, _ = np.histogram(p_vals, bins=bins)
    q_h, _ = np.histogram(q_vals, bins=bins)
    eps = 1e-8
    p = (p_h + eps) / (p_h.sum() + eps * n_bins)
    q = (q_h + eps) / (q_h.sum() + eps * n_bins)
    m = 0.5 * (p + q)
    jsd = 0.5 * np.sum(p * np.log2(p / m + 1e-12)) + \
          0.5 * np.sum(q * np.log2(q / m + 1e-12))
    return float(np.clip(jsd, 0.0, 1.0))

def compute_jsd_for_run(wrapper, tokenizer, model,
                         target_layer, target_expert,
                         code_texts, medical_texts, step):
    """Compute mean JSD across all spawned sub-adapters."""
    n_spawned = len(wrapper.spawn_loras[target_expert])
    if n_spawned == 0:
        return float('nan'), {}
    ca = _collect_gate_activations_qwen(wrapper, code_texts, tokenizer, model,
                                         target_layer, target_expert)
    ma = _collect_gate_activations_qwen(wrapper, medical_texts, tokenizer, model,
                                         target_layer, target_expert)
    scores = {}
    for j in range(n_spawned):
        scores[j] = _compute_jsd_scalar(ca.get(j, []), ma.get(j, []))
    valid = [v for v in scores.values() if not math.isnan(v)]
    mean_jsd = sum(valid) / len(valid) if valid else float('nan')
    jsd_str = "  ".join(f"sa{j}={v:.3f}" for j, v in scores.items())
    print(f"    [JSD @{step}] mean={mean_jsd:.3f}  |  {jsd_str}")
    return mean_jsd, scores

# ─────────────────────────────────────────────────────────────────────────────

TARGET_EXPERT = 5   # Expert 5, Layer 0 — highest conflict score for Qwen

all_results = {}
baseline_ppl_code = {}

for run_label, conflict_ratio in COMPARISON_CFG["conflict_ratios"].items():
    print("\n" + "=" * 80)
    print(f"RUN {run_label}: Conflict Ratio = {conflict_ratio:.0%}")
    print("=" * 80)

    encodings, domains = build_conflict_dataloader(
        tokenizer,
        conflict_ratio=conflict_ratio,
        max_length=COMPARISON_CFG["max_length"],
    )
    input_ids      = encodings["input_ids"]
    attention_mask = encodings["attention_mask"]
    n_samples = input_ids.shape[0]

    for method_name in COMPARISON_CFG["methods"]:
        run_key = f"{method_name}_{run_label}"
        _sep = "\u2500" * 70
        print(f"\n{_sep}")
        print(f"METHOD: {method_name.upper()} | RUN: {run_label}")
        print(_sep)

        gc.collect()
        torch.cuda.empty_cache()

        model_qwen = AutoModelForCausalLM.from_pretrained(
            MODEL_ID, torch_dtype=torch.bfloat16, device_map="cuda:0", trust_remote_code=True,
        )
        model_qwen.gradient_checkpointing_disable()
        model_qwen.config.use_cache = False

        wrapper, optimizer, extras = setup_method(model_qwen, method_name, COMPARISON_CFG)

        loss_log = []
        ppl_log  = []
        jsd_log  = []
        expansion_events = []

        initial_ppl = evaluate_perplexity(model_qwen, tokenizer, EVAL_CODE_TEXTS[:_N_EVAL])
        print(f"Initial code PPL: {initial_ppl:.2f}")
        ppl_log.append((0, initial_ppl))

        model_qwen.train()
        model_qwen.config.use_cache = False
        n_steps = COMPARISON_CFG["n_steps"]

        for step in range(n_steps):
            idx = step % n_samples
            batch_input = input_ids[idx:idx+1].to(model_qwen.device)
            batch_mask  = attention_mask[idx:idx+1].to(model_qwen.device)
            domain = domains[idx]

            labels = batch_input.clone()
            labels[:, :-1] = batch_input[:, 1:]
            labels[:, -1] = -100

            optimizer.zero_grad()
            outputs = model_qwen(input_ids=batch_input, attention_mask=batch_mask, labels=labels)
            loss = outputs.loss
            loss_log.append(loss.item())
            loss.backward()

            # Grad sanity check — confirm adapter grads are flowing
            if step % 100 == 0:
                for name, param in model_qwen.named_parameters():
                    if any(x in name for x in ["base_loras", "lora_A", "lora_B", "dr_loras"]):
                        if 'layers.0' in name and param.grad is not None and param.grad.abs().max() > 0:
                            print(f"  NONZERO GRAD: {name}: {param.grad.abs().mean().item():.6e}")
                            break
                else:
                    if step == 0:
                        print(f"  Step {step}: checking adapter grads...")

            # Method-specific post-backward
            if method_name == "hierarchical":
                n_spawned = len(wrapper.spawn_loras[TARGET_EXPERT])
                # Fixed-step spawning under conflict
                # TEST_MODE uses early steps (within n_steps=20); full run uses standard schedule
                _spawn_full = [150, 400, 650, 900, 1100]
                _spawn_test = [5, 12]  # within TEST_MODE n_steps=20
                spawn_at = (_spawn_test if TEST_MODE else _spawn_full) if conflict_ratio > 0 else []
                if step in spawn_at and n_spawned < COMPARISON_CFG["max_sub_adapters"]:
                    print(f"  [Step {step}] SPAWN: Expert {TARGET_EXPERT}")
                    lora = wrapper.base_loras[TARGET_EXPERT]
                    weight_grad = (lora.B.grad.float() @ lora.A.grad.float()
                                   if lora.B.grad is not None else None)
                    new_params = wrapper.spawn(TARGET_EXPERT, rank=8, weight_grad=weight_grad)
                    optimizer.add_param_group({'params': new_params, 'lr': COMPARISON_CFG["lr"]})
                    expansion_events.append((step, TARGET_EXPERT, "spawn"))

            elif method_name == "dr_lora":
                wrapper.tracker.update_routing_frequency_from_hooks()
                imp = wrapper.tracker.compute_rank_importance(wrapper.lora_modules)
                wrapper.tracker.update_rank_importance(imp)
                if wrapper.schedule.can_grow_at_step(step):
                    grow_res = perform_rank_growth(
                        tracker=wrapper.tracker, schedule=wrapper.schedule,
                        lora_modules=wrapper.lora_modules, current_step=step,
                        r_init=extras["r_init"], r_max=extras["r_max"],
                        p_grow=COMPARISON_CFG["p_grow"], gamma=COMPARISON_CFG["gamma"],
                    )
                    if grow_res.get("grew"):
                        wrapper.expanded = True
                        expansion_events.append((step, grow_res["num_experts_grown"], "rank_growth"))
                        print(f"  [Step {step}] RANK GROWTH: +{grow_res['total_new_ranks']} ranks")

            optimizer.step()

            if step % COMPARISON_CFG["log_every"] == 0:
                vram = torch.cuda.memory_allocated() / 1e9
                print(f"  Step {step:4d} | Loss: {loss.item():.4f} | Domain: {domain} | VRAM: {vram:.1f}GB")

            if (step + 1) % COMPARISON_CFG["eval_every"] == 0:
                ppl = evaluate_perplexity(model_qwen, tokenizer, EVAL_CODE_TEXTS[:_N_EVAL])
                ppl_log.append((step + 1, ppl))
                print(f"  [Step {step+1}] Code PPL: {ppl:.2f}")

                if method_name == "hierarchical" and len(wrapper.spawn_loras[TARGET_EXPERT]) > 0:
                    _med_probe = (_MED_CACHE or [])[:_N_EVAL]
                    if _med_probe:
                        mean_jsd, _ = compute_jsd_for_run(
                            wrapper, tokenizer, model_qwen,
                            COMPARISON_CFG["target_layer"], TARGET_EXPERT,
                            EVAL_CODE_TEXTS[:_N_EVAL], _med_probe, step + 1,
                        )
                        jsd_log.append((step + 1, mean_jsd))

        # Final eval
        final_ppl = evaluate_perplexity(model_qwen, tokenizer, EVAL_CODE_TEXTS[:_N_EVAL])
        print(f"\nFinal code PPL: {final_ppl:.2f}")
        print(f"Expansion events: {len(expansion_events)}")
        if jsd_log:
            valid_jsd = [(s, v) for s, v in jsd_log if not math.isnan(v)]
            if valid_jsd:
                peak_step, peak_jsd = max(valid_jsd, key=lambda x: x[1])
                print(f"Peak JSD: {peak_jsd:.3f} at step {peak_step}")

        adapter_state = {
            k: v.cpu().clone()
            for k, v in model_qwen.state_dict().items()
            if any(x in k for x in ["lora_A", "lora_B", "base_loras", "spawn_loras",
                                     "spawn_gate_", "rank_mask", ".A", ".B"])
        }
        print(f"  Adapter keys saved: {len(adapter_state)}")

        all_results[run_key] = {
            "method": method_name,
            "run": run_label,
            "conflict_ratio": conflict_ratio,
            "loss_log": loss_log,
            "ppl_log": ppl_log,
            "jsd_log": jsd_log,
            "final_ppl": final_ppl,
            "initial_ppl": initial_ppl,
            "expansion_events": expansion_events,
            "n_expansions": len(expansion_events),
            "adapter_state": adapter_state,
        }

        if run_label == "A":
            baseline_ppl_code[method_name] = final_ppl

        ckpt_path = os.path.join(SAVE_ROOT, f"qwen_{run_key}.pt")
        torch.save({
            "method": method_name, "run": run_label, "conflict_ratio": conflict_ratio,
            "loss_log": loss_log, "ppl_log": ppl_log, "jsd_log": jsd_log,
            "expansion_events": expansion_events,
            "adapter_state": adapter_state, "config": COMPARISON_CFG,
        }, ckpt_path)
        print(f"Saved: {ckpt_path}")

        del model_qwen
        gc.collect()
        torch.cuda.empty_cache()

print("\n" + "=" * 80)
print("CONFLICT-SCALING GRID COMPLETE")
print("=" * 80)

## Section 6 — Results


In [ ]:
# Results summary — identical to OLMoE (reads all_results dict)

print("\n" + "=" * 80)
print("THESIS RESULTS: CONFLICT-SCALING GRID  (Qwen1.5-MoE)")
print("=" * 80)
print(f"\n{'Method':<15} {'Run':<5} {'Conflict':<10} {'Code PPL':>10} {'Neg Transfer':>14} {'Expansions':>12}")
print("\u2500" * 75)

for run_label in ["A", "B", "C"]:
    for method_name in COMPARISON_CFG["methods"]:
        run_key = f"{method_name}_{run_label}"
        if run_key not in all_results:
            continue
        r = all_results[run_key]
        if run_label == "A":
            nt_str = "\u2014"
        else:
            baseline = baseline_ppl_code.get(method_name, r["final_ppl"])
            neg_transfer = r["final_ppl"] - baseline
            nt_str = f"{neg_transfer:+.2f}"
        print(f"{method_name:<15} {run_label:<5} {r['conflict_ratio']:.0%}{'':8} "
              f"{r['final_ppl']:>10.2f} {nt_str:>14} {r['n_expansions']:>12}")
    if run_label != "C":
        print()

print("\n" + "\u2500" * 75)
print("Neg Transfer = PPL(Run X) - PPL(Run A)  |  + means degraded  |  - means improved")
print("\nExpected: LoRA/DR-LoRA show increasing positive NegTransfer as conflict rises.")
print("          HR-LoRA spawns domain specialists \u2192 lower NegTransfer than baselines.")

In [ ]:
# Plots — full thesis figure set (identical to OLMoE, updated title)

import matplotlib.pyplot as plt
import math

colors = {"lora": "#2196F3", "dr_lora": "#4CAF50", "hierarchical": "#F44336"}
run_labels = ["A", "B", "C"]
methods = COMPARISON_CFG["methods"]

fig, axes = plt.subplots(2, 2, figsize=(16, 12))
fig.suptitle("HRLoRA vs Baselines \u2014 Qwen1.5-MoE Conflict-Scaling Grid",
             fontsize=14, fontweight="bold")

# Panel 1: NegTransfer bar chart
ax = axes[0, 0]
x = [0, 1]
bar_w = 0.22
for i, method_name in enumerate(methods):
    nts = []
    for rl in ["B", "C"]:
        ra = f"{method_name}_A"
        rx = f"{method_name}_{rl}"
        if ra in all_results and rx in all_results:
            nts.append(all_results[rx]["final_ppl"] - all_results[ra]["final_ppl"])
        else:
            nts.append(0)
    positions = [xi + i * bar_w for xi in x]
    bars = ax.bar(positions, nts, bar_w, label=method_name,
                  color=colors[method_name], alpha=0.85, edgecolor="white")
    for bar, nt in zip(bars, nts):
        ax.text(bar.get_x() + bar.get_width() / 2,
                bar.get_height() + 0.5,
                f"{nt:+.0f}", ha="center", va="bottom", fontsize=9, fontweight="bold")
ax.axhline(0, color="black", lw=0.8)
ax.set_xticks([xi + bar_w for xi in x])
ax.set_xticklabels(["Run B (20% conflict)", "Run C (50% conflict)"])
ax.set_ylabel("Negative Transfer (PPL increase vs Run A)")
ax.set_title("Negative Transfer by Method and Conflict Level")
ax.legend(); ax.grid(alpha=0.3, axis="y")

# Panel 2: PPL trajectories
ax = axes[0, 1]
for method_name in methods:
    for rl in ["B", "C"]:
        rk = f"{method_name}_{rl}"
        if rk not in all_results: continue
        r = all_results[rk]
        if not r["ppl_log"]: continue
        steps, ppls = zip(*r["ppl_log"])
        ls = "--" if rl == "C" else "-"
        ax.plot(steps, ppls, ls, color=colors[method_name], lw=1.8,
                label=f"{method_name} Run {rl}", alpha=0.85 if rl == "B" else 0.55)
ax.set_xlabel("Training Step"); ax.set_ylabel("Code Perplexity")
ax.set_title("PPL Trajectories (solid=B, dashed=C)")
ax.legend(fontsize=8); ax.grid(alpha=0.3)

# Panel 3: JSD over steps (hierarchical)
ax = axes[1, 0]
jsd_colors = {"B": "#E53935", "C": "#B71C1C"}
any_jsd = False
for rl in ["B", "C"]:
    rk = f"hierarchical_{rl}"
    if rk not in all_results: continue
    jsd_log = all_results[rk].get("jsd_log", [])
    valid = [(s, v) for s, v in jsd_log if not math.isnan(v)]
    if not valid: continue
    steps, vals = zip(*valid)
    ax.plot(steps, vals, "o-", color=jsd_colors[rl], lw=2, ms=5,
            label=f"Run {rl} (peak={max(vals):.3f})")
    any_jsd = True
if not any_jsd:
    ax.text(0.5, 0.5, "No JSD data\n(no spawns logged)", ha='center', va='center',
            transform=ax.transAxes, fontsize=12, color='gray')
ax.set_xlabel("Training Step"); ax.set_ylabel("Mean JSD of Gate Activations")
ax.set_title("Domain Specialization (JSD) — HR-LoRA only")
ax.legend(); ax.grid(alpha=0.3)

# Panel 4: Expansion events
ax = axes[1, 1]
bar_x, bar_labels, bar_colors = [], [], []
bi = 0
for method_name in methods:
    for rl in ["B", "C"]:
        rk = f"{method_name}_{rl}"
        if rk not in all_results: continue
        n_exp = all_results[rk]["n_expansions"]
        bar_x.append(n_exp)
        bar_labels.append(f"{method_name}\n{rl}")
        bar_colors.append(colors[method_name])
        bi += 1
if bar_x:
    bars = ax.bar(range(len(bar_x)), bar_x, color=bar_colors, alpha=0.85, edgecolor="white")
    ax.set_xticks(range(len(bar_x)))
    ax.set_xticklabels(bar_labels, fontsize=9)
    for bar, val in zip(bars, bar_x):
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.1,
                str(val), ha='center', va='bottom', fontsize=9)
ax.set_ylabel("Number of Expansions")
ax.set_title("Expansion Events (spawns for HR-LoRA, rank growth for DR-LoRA)")
ax.grid(alpha=0.3, axis="y")

plt.tight_layout()
fig_path = os.path.join(SAVE_ROOT, 'qwen_comparison_results.png')
plt.savefig(fig_path, dpi=150, bbox_inches='tight')
plt.show()
print(f"Saved: {fig_path}")

## Section 7 — HumanEval Evaluation


In [ ]:
!pip install human_eval

In [ ]:
# Inference utilities — adapted for Qwen

def load_fresh_model():
    """Load clean Qwen base model for inference. use_cache=True always."""
    gc.collect(); torch.cuda.empty_cache()
    free, total = torch.cuda.mem_get_info()
    print(f"  VRAM before load: {free/1e9:.2f} GB free / {total/1e9:.2f} GB total")
    assert free > 5e9, f"Not enough VRAM ({free/1e9:.2f} GB)"
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID, torch_dtype=torch.bfloat16, device_map="cuda:0", trust_remote_code=True,
    )
    model.config.use_cache = True   # always True for inference
    return model


def load_adapter_weights(model, adapter_state):
    """Load adapter state dict and report what loaded."""
    model_keys = {k for k, v in model.named_parameters()
                  if any(x in k for x in ["lora_A", "lora_B", "base_loras",
                                           "spawn_loras", "spawn_gate_", ".A", ".B"])}
    saved_keys = set(adapter_state.keys())
    matched    = model_keys & saved_keys
    missing    = model_keys - saved_keys
    unexpected = saved_keys - model_keys

    print(f"  Adapter keys — saved: {len(saved_keys)}, model: {len(model_keys)}, matched: {len(matched)}")
    if missing:
        print(f"  \u26a0 Missing: {len(missing)}")
        for k in sorted(missing)[:3]: print(f"      {k}")
    if unexpected:
        print(f"  \u26a0 Unexpected: {len(unexpected)}")
        for k in sorted(unexpected)[:3]: print(f"      {k}")
    if not matched:
        print("  \u2717 FATAL: zero keys matched"); return False

    model.load_state_dict(adapter_state, strict=False)

    # Rebuild spawn gate registry — plain Python dict, not serialized in state_dict
    for layer in model.model.layers:
        experts = getattr(layer.mlp, 'experts', None)
        if isinstance(experts, HierarchicalQwenExperts):
            experts.rebuild_gate_registry()
            n = sum(len(v) for v in experts.spawn_loras)
            print(f"  Rebuilt gate registry: {n} spawned adapters restored")

    b_keys = [k for k in matched if k.endswith(".B") or "lora_B" in k]
    nontrivial = sum(1 for k in b_keys[:10]
                     if dict(model.named_parameters()).get(k, torch.zeros(1)).abs().max().item() > 1e-6)
    _chk = "\u2713" if nontrivial > 0 else "\u2717 weights not loaded"
    print(f"  B-matrix check: {nontrivial}/{min(len(b_keys),10)} non-zero ({_chk})")
    return nontrivial > 0 or len(b_keys) == 0


def generate_completion(model, tokenizer, prompt, max_new_tokens=512):
    model.config.use_cache = True
    model.eval()
    inputs = tokenizer(prompt, return_tensors='pt', truncation=True,
                       max_length=1024).to(model.device)
    with torch.no_grad():
        outputs = model.generate(
            inputs.input_ids, max_new_tokens=max_new_tokens,
            do_sample=False, use_cache=True,
            pad_token_id=tokenizer.pad_token_id,
        )
    return tokenizer.decode(outputs[0][len(inputs.input_ids[0]):], skip_special_tokens=True)


def sanity_check(model, tokenizer, label):
    """Quick generation sanity check."""
    prompt = "def add(a, b):\n    "
    completion = generate_completion(model, tokenizer, prompt, max_new_tokens=30)
    print(f"  [{label}] prompt: '{prompt.strip()}' → '{completion[:60]}'")

print("Inference utilities defined.")

In [ ]:
# HumanEval evaluation — identical to OLMoE

from human_eval.data import read_problems
from human_eval.evaluation import evaluate_functional_correctness

def run_humaneval(model, tokenizer, label):
    problems = read_problems()
    print(f"\n[{label}] Running HumanEval on {len(problems)} problems...")
    samples = []
    for i, (task_id, problem) in enumerate(problems.items()):
        completion = generate_completion(model, tokenizer, problem['prompt'])
        samples.append({'task_id': task_id, 'completion': completion})
        if (i + 1) % 20 == 0:
            print(f"  Generated {i+1}/{len(problems)}")
    samples_path = os.path.join(SAVE_ROOT, f'humaneval_{label}.jsonl')
    with open(samples_path, 'w') as f:
        for s in samples:
            f.write(json.dumps(s) + '\n')
    results = evaluate_functional_correctness(samples_path)
    pass_at_1 = results['pass@1'] * 100
    print(f"[{label}] Pass@1: {pass_at_1:.2f}%")
    return pass_at_1


humaneval_results = {}

# Base model
print("=" * 60)
print("EVALUATING: base model (no adapter)")
print("=" * 60)
model_eval = load_fresh_model()
sanity_check(model_eval, tokenizer, "base")
# humaneval_results["base"] = run_humaneval(model_eval, tokenizer, "base")
del model_eval; gc.collect(); torch.cuda.empty_cache()

# Each method, Run C adapter
for method_name in COMPARISON_CFG["methods"]:
    run_key = f"{method_name}_C"
    print(f"\n{'='*60}\nEVALUATING: {method_name} Run C\n{'='*60}")

    if run_key not in all_results or "adapter_state" not in all_results[run_key]:
        print(f"  Skipping — {run_key} not in all_results"); continue

    model_eval = load_fresh_model()
    try:
        wrapper, _, _ = setup_method(model_eval, method_name, COMPARISON_CFG)
        loaded_ok = load_adapter_weights(model_eval, all_results[run_key]["adapter_state"])
        if not loaded_ok:
            print(f"  \u2717 Adapter load failed — skipping HumanEval"); continue
        sanity_check(model_eval, tokenizer, f"{method_name}_C")
        # humaneval_results[method_name] = run_humaneval(model_eval, tokenizer, method_name)
    except Exception as e:
        import traceback; traceback.print_exc()
    finally:
        del model_eval; gc.collect(); torch.cuda.empty_cache()

print("\n" + "=" * 60)
print("HUMANEVAL SUMMARY")
print("=" * 60)
for label, score in humaneval_results.items():
    print(f"  {label:<20} Pass@1: {score:.2f}%")

In [ ]:
# Save complete experiment summary

summary = {
    "model": MODEL_ID,
    "config": COMPARISON_CFG,
    "n_total_per_run": N_TOTAL,
    "ppl_results": {},
    "negative_transfer": {},
    "humaneval": humaneval_results,
}

for run_key, r in all_results.items():
    m = r["method"]; rl = r["run"]
    if m not in summary["ppl_results"]:
        summary["ppl_results"][m] = {}
    summary["ppl_results"][m][rl] = {
        "conflict_ratio": r["conflict_ratio"],
        "final_ppl": r["final_ppl"],
        "n_expansions": r["n_expansions"],
    }
    if rl != "A" and m in baseline_ppl_code:
        if m not in summary["negative_transfer"]:
            summary["negative_transfer"][m] = {}
        summary["negative_transfer"][m][rl] = r["final_ppl"] - baseline_ppl_code[m]

summary_path = os.path.join(SAVE_ROOT, 'qwen_experiment_summary.json')
json_safe = {k: {kk: vv for kk, vv in v.items() if kk != "adapter_state"}
             for k, v in all_results.items()}
summary["all_results"] = json_safe
with open(summary_path, 'w') as f:
    json.dump(summary, f, indent=2, default=str)
print(f"Summary saved to: {summary_path}")
print("\nDone. \u2705")